<!-- nav -->
[← 4 · Constraints that take arguments](04_Constraints_That_Take_Arguments.ipynb) · [Home](../../README.md) · [6 · Drawing values →](06_Drawing_Values.ipynb)

# 5 · The shape of a population

**The problem.** Test data, sample data, synthetic data: each needs not just valid contacts but a believable *mix* of
them. Three in ten are seniors; most have a phone. A validator can't express "most": no one contact breaks it. The mix
is a property of the **population**.

A **pattern** is a predicate whose constraint also weighs alternatives. `Distributions.Choices` holds weighted **arms**;
the pattern is still a predicate, so it still checks and selects, and now it also weighs.

In [1]:
from collections import Counter

from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Distributions, Generators, Predicates, Validators
from mbse.Schemas.Framework import Stores
from toolkit import Contact, Directory, book, listed

person = E.variable("person")
HasName = (Predicates.OfPredicate.Builder().name("HasName").symbols({"person": Contact})
           .parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())

## Step 1: weighted alternatives

Each arm has a weight and conditions. The choices hold when enough arms hold: at least one, by default.

In [2]:
Names = (Predicates.OfPredicate.Builder().name("Names").symbols({"person": Contact}).requires(
    Distributions.Choices.Builder().arms(
        lambda a: a.weight(10).requires(HasName(person, "Ann")),
        lambda a: a.weight(5).requires(HasName(person, "Bob")),
        lambda a: a.weight(15).requires(HasName(person, "Cy")),
    ).create()).create())
store = book()
listed(store, *(store.Contact().name(n).create() for n in ("Ann", "Bob", "Cy", "Zed")))
Validators.Validate(store, [Names]).Reachable(Directory, store.singleton("book.Directory"))

["person=Contact#4: 'Names' does not hold"]

As a constraint, the pattern says "the name is one of these", and Zed breaks it.

## Step 2: what a match weighs

`Evaluator.weigh` gives what a match weighs under the constraint: nothing unless it holds, then its arm's weight.

In [3]:
evaluate = Predicates.Evaluator(store)
{c.name: evaluate.weigh(Names.requires, {"person": c}) for c in store.extent("Contact")}

{'Ann': 10.0, 'Bob': 5.0, 'Cy': 15.0, 'Zed': 0.0}

## Step 3: counts and precedence

`.count(...)` says how many arms must hold, as a condition on the number. `.decreasing()` puts the arms in order: a
match falls under the first that holds, so put the more specific arms first.

In [4]:
Overlap = Distributions.Choices.Builder().arms(
    lambda a: a.weight(3).requires(person.has("name")),
    lambda a: a.weight(2).requires(HasName(person, "Ann")))
ann = next(c for c in store.extent("Contact") if c.name == "Ann")
(evaluate.weigh(Overlap.create(), {"person": ann}),               # every arm that holds: 3 + 2
 evaluate.weigh(Overlap.decreasing().create(), {"person": ann}),  # the first that holds
 evaluate(Distributions.Choices.Builder().arms(lambda a: a.weight(1).requires(person.has("name")),
                                               lambda a: a.weight(1).requires(HasName(person, "Ann")))
          .count(lambda c: c == 1).create(), {"person": ann}))   # exactly one arm: not Ann, who holds both

(5.0, 3.0, False)

## Step 4: sampling the store

`Generators.Sample` draws the store's own matches, each as often as it weighs, from a random source you give it. The
same seed gives the same draws, in Python and TypeScript alike:

In [5]:
drawn = Generators.Sample(store, Names, Stores.PCG32(42))
Counter(next(drawn)["person"].name for _ in range(3000))

Counter({'Cy': 1513, 'Ann': 1004, 'Bob': 483})

About 10 : 5 : 15, and never Zed.

## Why it works this way

- **A pattern is a predicate.** It validates, queries and weighs with one constraint, so the data you generate from it
  and the data you check against it can't disagree.
- **Weights are relative.** Arms needn't sum to one: 10, 5 and 15 are the same mix as 1/3, 1/6 and 1/2.
- **The random source is the caller's.** A store is data access alone; seeds belong to whoever draws.

<!-- nav -->
[← 4 · Constraints that take arguments](04_Constraints_That_Take_Arguments.ipynb) · [Home](../../README.md) · [6 · Drawing values →](06_Drawing_Values.ipynb)